In [1]:
import os
import json
import pandas as pd
from tqdm import tqdm

# Sort JSON files in numeric order like 1.json, 2.json, ..., 6.json
json_files = sorted(
    [f for f in os.listdir('.') if f.endswith('.json')],
    key=lambda x: int(os.path.splitext(x)[0])
)

all_hashes = []
total_rows = 0
total_files = 0

for filename in tqdm(json_files, desc="Processing JSON files in order"):
    with open(filename, 'r') as f:
        try:
            data = json.load(f)
            rows = data.get("result", {}).get("rows", [])
            total_rows += len(rows)
            hashes = [row["hash"] for row in rows if "hash" in row]
            all_hashes.extend(hashes)
            total_files += 1
        except Exception as e:
            print(f"Error reading {filename}: {e}")

print(f"Processed {total_files} files, found {total_rows} rows, extracted {len(all_hashes)} hashes")

if all_hashes:
    df = pd.DataFrame(all_hashes, columns=["hash"])
    df.to_parquet("all_hashes.parquet", index=False)
    print("Parquet file written.")
else:
    print("No hashes extracted. Check JSON structure.")


Processing JSON files in order: 0it [00:00, ?it/s]

Processed 0 files, found 0 rows, extracted 0 hashes
No hashes extracted. Check JSON structure.


In [2]:
import pyarrow.parquet as pq

table = pq.read_table("all_hashes.parquet", columns=["hash"])
last_hash = table[-1].to_pandas()
#print(last_hash)

In [3]:
last_row = table.slice(len(table) - 1, 1).to_pandas()
print(last_row.to_string(index=False))

                                                              hash
0x86edd762dbb09c3995ab6078e94f06347bb977b7a60c72eef509db5228125eb9


In [4]:
# get the first row
first_row = table.slice(0, 1).to_pandas()
print(first_row.to_string(index=False))

                                                              hash
0x8eb37e1a96125474123758cd8b0e82dcb2a03f482915a93b4e21a51bcee19c83


In [6]:
# Let's see how big this parquet file is (how many items)
print(f"Total hashes in parquet file: {len(table)}")

Total hashes in parquet file: 6313121
